In [3]:
import scipy.io as sio
import numpy as np

# Load subject 1, exercise 1
data = sio.loadmat('data/DB5_s1/S1_E1_A1.mat')
print(data.keys())  # see what's inside

emg = data['emg']        # shape: (samples, 16 channels)
labels = data['restimulus']  # gesture label per sample
print(emg.shape, labels.shape)

dict_keys(['__header__', '__version__', '__globals__', 'emg', 'acc', 'stimulus', 'glove', 'subject', 'exercise', 'repetition', 'restimulus', 'rerepetition', 'age', 'circumference', 'frequency', 'gender', 'height', 'weight', 'laterality', 'sensor'])
(130267, 16) (130267, 1)


In [4]:
from scipy.signal import butter, filtfilt

def bandpass_filter(signal, lowcut=20, highcut=450, fs=200, order=4):
    nyq = fs / 2
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal, axis=0)

def extract_features(window):
    # 4 classic EMG features per channel
    mav = np.mean(np.abs(window), axis=0)        # mean absolute value
    rms = np.sqrt(np.mean(window**2, axis=0))    # root mean square
    zc = np.sum(np.diff(np.sign(window), axis=0) != 0, axis=0)  # zero crossings
    wl = np.sum(np.abs(np.diff(window, axis=0)), axis=0)        # waveform length
    return np.concatenate([mav, rms, zc, wl])   # 64 features total (4 × 16 channels)

def build_dataset(emg, labels, window_size=200, step=100):
    X, y = [], []
    filtered = bandpass_filter(emg)
    for i in range(0, len(filtered) - window_size, step):
        window = filtered[i:i+window_size]
        label = labels[i + window_size//2][0]
        if label == 0:  # skip rest gesture
            continue
        X.append(extract_features(window))
        y.append(label)
    return np.array(X), np.array(y)

emg_filtered = bandpass_filter(emg)
X, y = build_dataset(emg, labels)
print(f"Dataset: {X.shape}, Labels: {np.unique(y)}")

ValueError: Digital filter critical frequencies must be 0 < Wn < 1